# Reranker `bge-reranker-v2-m3` con GPU (Colab T4) — RAG Banco de Occidente

Evalúa **búsqueda híbrida + filtro de vigencia + reranker bge (top-20)** sobre el golden set, para uno o varios modelos de embeddings, con la misma lógica del repositorio
(`pipeline/15_embeddings_con_reranker.py`). En CPU esta evaluación tarda cerca de 1,5 horas; en una T4 debería tardar unos minutos (estimación, no medida).

**Carpeta en Drive: `rag-bdo/`, con la misma estructura del repositorio**
```
MyDrive/rag-bdo/
├── rag-bocc/corpus.jsonl                       (se regenera con: python pipeline/20_reconstruir_corpus.py)
├── rag-bocc/golden/golden-v0.jsonl
├── pipeline/15_embeddings_con_reranker.py
├── pipeline/fase3a/  chunking.py  retrieval.py  evaluar.py  vigencia.py  almacen.py
└── _raw/emb/<modelo>/                          solo los modelos que vayas a evaluar:
      e5-small/            estructural__ctx1.npy  .ids.json  .hashes.json        (las consultas se codifican aquí mismo)
      voyage-context-4/    estructural__ctx1.npy  .ids.json  .hashes.json  consultas.npz
      voyage-4-large/      (igual que el anterior)
```
Los vectores deben estar **al día con el corpus** (si no, el cuaderno se detiene y te dice cuántos faltan; recalcúlalos con `embeddings_colab.ipynb` o `13_embeddings_api.py`).
Resultado: `rag-bdo/rag-bocc/evaluacion/fase3a/embeddings-reranker-dev<sufijo>.json`. Activa la GPU: *Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*.

In [ ]:
!pip -q install sentence-transformers
import torch
print('CUDA disponible:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'sin GPU (será muy lento)')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
RAIZ = '/content/drive/MyDrive/rag-bdo'
os.environ['RAG_BASE'] = RAIZ
os.makedirs(f'{RAIZ}/rag-bocc/evaluacion/fase3a', exist_ok=True)

In [ ]:
# Configuración
MODELOS = ['e5-small', 'voyage-context-4', 'voyage-4-large']    # los que quieras comparar
SPLIT_FIJO = 'dev'                                                # 'test' solo UNA vez con el modelo ya elegido
SUFIJO = '-colab'                                                 # evita pisar resultados locales
USAR_FP16 = True                                                  # más rápido en T4; ordena igual salvo diferencias mínimas

In [ ]:
# Comprobar archivos
req = ['rag-bocc/corpus.jsonl', 'rag-bocc/golden/golden-v0.jsonl', 'pipeline/15_embeddings_con_reranker.py'] + [f'pipeline/fase3a/{f}.py' for f in ['chunking', 'retrieval', 'evaluar', 'vigencia', 'almacen']]
for m in MODELOS:
    req += [f'_raw/emb/{m}/estructural__ctx1.{e}' for e in ('npy', 'ids.json', 'hashes.json')]
    if m != 'e5-small': req.append(f'_raw/emb/{m}/consultas.npz')
faltan = [f for f in req if not os.path.exists(f'{RAIZ}/{f}')]
print('Todo en su sitio' if not faltan else 'FALTAN:\n  ' + '\n  '.join(faltan))

In [ ]:
# Ejecutar la evaluación (guarda el resultado en Drive aunque se corte la sesión después)
!python {RAIZ}/pipeline/15_embeddings_con_reranker.py --modelos {' '.join(MODELOS)} --split {SPLIT_FIJO} --sufijo={SUFIJO} --dispositivo cuda {'--fp16' if USAR_FP16 else ''} --lote 32

In [ ]:
# Resumen
import json
r = json.load(open(f'{RAIZ}/rag-bocc/evaluacion/fase3a/embeddings-reranker-{SPLIT_FIJO}{SUFIJO}.json'))
print(f"{'modelo':20s} {'cita@1':>7s} {'cita@10':>8s} {'MRR':>6s} | {'var cita@1':>10s} {'var cita@10':>11s} {'var MRR':>8s}")
for m, x in r.items():
    a, v = x['todos|pregunta'], x['todos|variante']
    print(f"{m:20s} {a['cita@1']:7.3f} {a['cita@10']:8.3f} {a['mrr']:6.3f} | {v['cita@1']:10.3f} {v['cita@10']:11.3f} {v['mrr']:8.3f}")